# N-Grams
### Practical Assignment - Advanced Machine Learning - Sebastiaan Craens and Ivan Runov

We first created a notebook for generating the N-grams. We quickly learned that the data structure we used at the time was not going to cut it.
We first used a dictionary of counts where the key was a tuple of the n-gram strings. Then we switched to using indices from a str->int mapping instead of strings which resulted in a noticable speedup and a significant memory reduction. We also changed the data structure to where the key is the dict is a tuple of the tokens except for the last one, with the values being dictionaries of the last words in the n-grams with counts.

By iteratively generating sentences, we surmised that in this case, 4-grams produced the best results. We eventually settled on using sentence ending tokens to start the sentences as wel as to end them. We also tweaked our preprocessing a couple of times based on the results.

In [2]:
import os
import gzip
import pickle
import numpy as np

### Settings

In [3]:
DIR_DATA = "data"
DIR_TEXTS = os.path.join(DIR_DATA, "texts")
FILE_METADATA = os.path.join(DIR_DATA, "metadata.csv")

np.random.seed(67)

In [4]:
# Load ngrams
MAX_N = 5
NGRAM_SOURCE = "idx"
NGRAM_SOURCE_FILE = f"ngrams_{MAX_N}_{NGRAM_SOURCE}.pkl"
with open(os.path.join(DIR_DATA, NGRAM_SOURCE_FILE), "rb") as f:
    ngrams = pickle.load(f)
    
print(f"Loaded {', '.join(map(str, range(2, MAX_N + 1)))}-grams ({NGRAM_SOURCE}) from {os.path.join(DIR_DATA, NGRAM_SOURCE_FILE)}.")

Loaded 2, 3, 4, 5-grams (idx) from data\ngrams_5_idx.pkl.


In [5]:
# Load dictionary mappings
INDEX_TO_TOKEN_FILE = f"index_to_token.pkl"
TOKEN_TO_INDEX_FILE = f"token_to_index.pkl"

with open(os.path.join(DIR_DATA, INDEX_TO_TOKEN_FILE), "rb") as f:
    idx_to_token = pickle.load(f)

with open(os.path.join(DIR_DATA, TOKEN_TO_INDEX_FILE), "rb") as f:
    token_to_idx = pickle.load(f)

print(f"Loaded index-to-token mapping from {os.path.join(DIR_DATA, INDEX_TO_TOKEN_FILE)}.")
print(f"Loaded token-to-index mapping from {os.path.join(DIR_DATA, TOKEN_TO_INDEX_FILE)}.")

Loaded index-to-token mapping from data\index_to_token.pkl.
Loaded token-to-index mapping from data\token_to_index.pkl.


In [6]:
sorted_ngrams = {
    n: sorted(ngrams[n].items(), key=lambda item: sum(item[1].values()), reverse=True) for n in range(2, MAX_N + 1)
}
print(f"Sorted n-grams by frequency.")

Sorted n-grams by frequency.


In [7]:
print(f"Most common ngrams:")
for n in range(2, MAX_N + 1):
    print(f"  {n}-grams:")
    # n_gram is now a tuple of IDs
    for n_gram_ids, follow_ups in sorted_ngrams[n][:10]:
        words = [idx_to_token[i] for i in n_gram_ids]  # Get the n-gram IDs
        most_common_follow_up_id = max(follow_ups, key=follow_ups.get)
        most_common_count = follow_ups[most_common_follow_up_id]
        print(f"    \"{' '.join(words)}\": {idx_to_token[most_common_follow_up_id]} ({most_common_count} occurrences)")

Most common ngrams:
  2-grams:
    "the": and (26320 occurrences)
    ".": i (41121 occurrences)
    "and": the (28102 occurrences)
    "of": the (86746 occurrences)
    "to": the (41987 occurrences)
    "a": and (7767 occurrences)
    "in": the (62468 occurrences)
    "i": have (12472 occurrences)
    "he": was (22215 occurrences)
    "that": he (13080 occurrences)
  3-grams:
    "of the": and (4569 occurrences)
    "in the": and (2749 occurrences)
    "to the": and (2397 occurrences)
    ". i": am (2973 occurrences)
    ". the": first (582 occurrences)
    ". he": was (6126 occurrences)
    "and the": and (614 occurrences)
    "on the": and (1238 occurrences)
    "the and": the (2909 occurrences)
    "to be": a (1436 occurrences)
  4-grams:
    ". he was": a (768 occurrences)
    ". it was": a (890 occurrences)
    "of the and": the (578 occurrences)
    ". it is": a (458 occurrences)
    "one of the": most (397 occurrences)
    "dau . of": the (809 occurrences)
    ". in the": first

In [8]:
print(f"Most common ngrams excluding sentence punctuation:")
for n in range(2, MAX_N + 1):
    print(f"  {n}-grams:")
    i = 0
    # Assuming sorted_ngrams[n] is used here as it's defined in cell #VSC-7a12bfb2
    for n_gram_ids, follow_ups in sorted_ngrams[n]:
        words = [idx_to_token[i] for i in n_gram_ids]  # Get the n-gram IDs
        most_common_follow_up_id = max(follow_ups, key=follow_ups.get)
        most_common_count = follow_ups[most_common_follow_up_id]
        
        if any(word in {".", "!", "?"} for word in words):
            continue
        
        print(f"    \"{' '.join(words)}\": {idx_to_token[most_common_follow_up_id]} ({most_common_count} occurrences)")
        i += 1
        if i >= 10:
            break

Most common ngrams excluding sentence punctuation:
  2-grams:
    "the": and (26320 occurrences)
    "and": the (28102 occurrences)
    "of": the (86746 occurrences)
    "to": the (41987 occurrences)
    "a": and (7767 occurrences)
    "in": the (62468 occurrences)
    "i": have (12472 occurrences)
    "he": was (22215 occurrences)
    "that": he (13080 occurrences)
    "was": a (11774 occurrences)
  3-grams:
    "of the": and (4569 occurrences)
    "in the": and (2749 occurrences)
    "to the": and (2397 occurrences)
    "and the": and (614 occurrences)
    "on the": and (1238 occurrences)
    "the and": the (2909 occurrences)
    "to be": a (1436 occurrences)
    "he was": a (1747 occurrences)
    "at the": same (1626 occurrences)
    "it was": a (1949 occurrences)
  4-grams:
    "of the and": the (578 occurrences)
    "one of the": most (397 occurrences)
    "out of the": and (223 occurrences)
    "the and the": and (154 occurrences)
    "in the and": the (253 occurrences)
    "to t

In [9]:
print(f"Number of occurences of some bigrams:")
for bigram_words in [("the", "cat"), ("him", "!"), ("her", "!"), ("him", "?", ), ("her", "?")]:
    try:
        # Convert words to IDs before lookup
        bigram_ids = tuple(token_to_idx[word] for word in bigram_words)
        count = ngrams[len(bigram_words)].get(bigram_ids[:-1], {}).get(bigram_ids[-1], 0)
    except KeyError:
        # If any word in bigram is not in vocab
        count = 0
    print(f"    {bigram_words}: {count}")

Number of occurences of some bigrams:
    ('the', 'cat'): 176
    ('him', '!'): 428
    ('her', '!'): 229
    ('him', '?'): 583
    ('her', '?'): 323


In [ ]:
# # Most common words that end sentences
# print(f"Most common words that end sentences:")
# sentence_endings_set = {".", "!", "?"}

# ending_counts = {
#     punctuation: {} for punctuation in sentence_endings_set
# }

# # The data structure in ngrams[2] is now {(id1, id2): count}
# for bigram_ids, count in ngrams[2].items():
#     # Convert punctuation ID back to string to check if it's an ending
#     punctuation_str = idx_to_token[bigram_ids[1]]
    
#     if punctuation_str in sentence_endings_set:
#         word_str = idx_to_token[bigram_ids[0]]
#         ending_counts[punctuation_str][word_str] = ending_counts[punctuation_str].get(word_str, 0) + count
        
# for punctuation, words in ending_counts.items():
#     print(f"  {punctuation}:")
#     for word, count in sorted(words.items(), key=lambda x: x[1], reverse=True)[:10]:
#         print(f"    {word}: {count}")

Most common words that end sentences:


IndexError: tuple index out of range

In [11]:
# Most common words that start sentences by punctuation
print(f"Most common words that start sentences by punctuation:")
sentence_startings_set = {".", "!", "?"}

starting_counts = {
    punctuation: {} for punctuation in sentence_startings_set
}

# The data structure in ngrams[2] is now {(id1, id2): count}
for bigram_ids, count in ngrams[2].items():
    # Check punctuation (first element of bigram)
    punctuation_str = idx_to_token[bigram_ids[0]]
    
    if punctuation_str in sentence_startings_set:
        word_str = idx_to_token[bigram_ids[1]]
        starting_counts[punctuation_str][word_str] = starting_counts[punctuation_str].get(word_str, 0) + count
        
for punctuation, words in starting_counts.items():
    print(f"  {punctuation}:")
    for word, count in sorted(words.items(), key=lambda x: x[1], reverse=True)[:10]:
        print(f"    {word}: {count}")

Most common words that start sentences by punctuation:


IndexError: tuple index out of range

### Word prediction

In [12]:
# Calculate ngram probabilities
ngram_totals = {
    n: {
        context: sum(next_tokens.values()) 
        for context, next_tokens in ngrams[n].items()
    } for n in range(2, MAX_N + 1)
}
ngram_probs = {
    n: {
        context: {token: count / ngram_totals[n][context] for token, count in next_tokens.items()}
        for context, next_tokens in ngrams[n].items()
    } for n in range(2, MAX_N + 1)
}

In [13]:
# Filter ngrams to exclude those with only one next token (i.e., deterministic continuations)
filtered_ngrams = {
    n: {
        context: next_tokens for context, next_tokens in ngrams[n].items() if len(next_tokens) > 1
    } for n in range(2, MAX_N + 1)
}

In [14]:
END_OF_SENTENCE_WORDS = [".", "!", "?"]
END_OF_SENTENCE_TOKENS = {token_to_idx[word] for word in END_OF_SENTENCE_WORDS}

MAX_WORDS_TO_GENERATE = 10

import time

def generate_sentence_smart(ngrams, n: int, runs: int):
    print(f"Generating with {n}-grams:")
    
    n_keys = list(ngrams[n].keys())
    
    for run_i in range(runs):
        
        # # Take random n-gram as starting point
        # start_idx = np.random.choice(range(len(n_keys)))
        # sentence = n_keys[start_idx]
        
        # Take random sentence ending as starting point
        sentence_idx = np.random.choice(len(END_OF_SENTENCE_TOKENS))
        sentence = (tuple([list(END_OF_SENTENCE_TOKENS)[sentence_idx]]))
        
        print(f"  Run {run_i + 1}:", end = "")
        
        # Add words until we have n-1 tokens to start with
        while len(sentence) < n - 1:
            keys = list(ngrams[len(sentence)+1][sentence].keys())
            next_token_idx = np.random.choice(range(len(keys)), p=list(ngram_probs[len(sentence)+1][sentence].values()))
            
            
            next_token = keys[next_token_idx]
            sentence = sentence + (next_token,)
            
            word = idx_to_token[next_token]
            if len(sentence) == 1:  # First word after initial context
                word = word.capitalize()
            print(" " + word, end = "")
        
        # print(f"  Run {run_i + 1}: {' '.join(idx_to_token[i] for i in sentence)}", end = "")
        
        
        # for word_i in range(MAX_WORDS_TO_GENERATE):
        while True:
            
            # time.sleep(0.01)
             
            # Pick one based on frequency
            last_tokens = sentence[-(n-1):]
            keys = list(ngrams[n][tuple(last_tokens)].keys())
            next_token_id = np.random.choice(range(len(keys)), p=list(ngram_probs[n][tuple(last_tokens)].values()))
            next_token = keys[next_token_id]
            
            # Update sequence with new token
            sentence = sentence + (next_token,)
                        
            # if not next_token in END_OF_SENTENCE_TOKENS and word_i < MAX_WORDS_TO_GENERATE - 1:
            # if word_i < MAX_WORDS_TO_GENERATE - 1:
            if not next_token in END_OF_SENTENCE_TOKENS:
                word = idx_to_token[next_token]
                # Capitalise if first
                if( len(sentence) == n - 1):  # First word after initial context
                    word = word.capitalize()                
                
                print(" " + word, end = "")
            
            else:
                print(idx_to_token[next_token])
                break
            
            
for n in range(2, MAX_N + 1):
    generate_sentence_smart(ngrams, n, 5)

Generating with 2-grams:
  Run 1: colevile is something in your do harm would have loved her without taking the street.
  Run 2: at the helmet thundering his golden buckles.
  Run 3: seek her hypnotic influence of our knowing that and promoted lieut.
  Run 4: i frequently means will you never mind.
  Run 5: it will as it the bending their and after powder under the only of his baser matter of my and scolded the diviner joy to those my liver pours.
Generating with 3-grams:
  Run 1: i have been his.
  Run 2: do you mean been thinking of something that anne had her own they will will will fulfil all his views plainly to tell do not no one saw her very description of this isle.
  Run 3: if thou wear better than the office of wherein i upon thy tongue with he laboriously pursing his lips.
  Run 4: and what is death.
  Run 5: how?
Generating with 4-grams:
  Run 1: those two are gone?
  Run 2: an we should give you our voices heartily.
  Run 3: scarcely had it reached ere the though still as 

In [ ]:
# def generate_sentence_dumb(ngrams, n: int, runs: int):
#     print(f"Generating with {n}-grams:")
    
#     n_keys = list(ngrams[n].keys())
    
#     for run_i in range(runs):
        
#         # # Take random n-gram as starting point
#         # start_idx = np.random.choice(range(len(n_keys)))
#         # sentence = n_keys[start_idx]
        
#         # Take random sentence ending as starting point
#         sentence_idx = np.random.choice(len(END_OF_SENTENCE_TOKENS))
#         sentence = (tuple([list(END_OF_SENTENCE_TOKENS)[sentence_idx]]))
        
#         # print(f"  Run {run_i + 1}:", end = "")
        
#         # Start with a random n-gram as starting point
#         start_idx = np.random.choice(range(len(n_keys)))
#         sentence = n_keys[start_idx]
#         print(f"  Run {run_i + 1}: {' '.join(idx_to_token[i] for i in sentence)}", end = "")
        
#         # print(f"  Run {run_i + 1}: {' '.join(idx_to_token[i] for i in sentence)}", end = "")
        
        
#         # for word_i in range(MAX_WORDS_TO_GENERATE):
#         while True:
            
#             # time.sleep(0.01)
             
#             # Pick one based on frequency
#             last_tokens = sentence[-(n-1):]
#             keys = list(ngrams[n][tuple(last_tokens)].keys())
#             next_token_id = np.random.choice(range(len(keys)), p=list(ngram_probs[n][tuple(last_tokens)].values()))
#             next_token = keys[next_token_id]
            
#             # Update sequence with new token
#             sentence = sentence + (next_token,)
                        
#             # if not next_token in END_OF_SENTENCE_TOKENS and word_i < MAX_WORDS_TO_GENERATE - 1:
#             # if word_i < MAX_WORDS_TO_GENERATE - 1:
#             if not next_token in END_OF_SENTENCE_TOKENS:
#                 word = idx_to_token[next_token]
#                 # Capitalise if first
#                 if( len(sentence) == n - 1):  # First word after initial context
#                     word = word.capitalize()                
                
#                 print(" " + word, end = "")
            
#             else:
#                 print(idx_to_token[next_token])
#                 break

In [ ]:
# for n in range(2, MAX_N + 1):
#     generate_sentence_dumb(filtered_ngrams, n , 5)

Generating with 2-grams:
  Run 1: whom, then, and was lost when he very reverend sir, though richard.
  Run 2: diligent application, he walked along the courts of the end of our homely proverb, “that if we waited for the light of cut her and wealth went to abate of time, however, gines stole down to them, and dreading that part of jumping about ceres.
  Run 3: peaceably vowed.
  Run 4: bad.” “he’s just the negro man, “but rosamond was ungenerous.
  Run 5: 1893, to our undertaking a sort of natural action into the loss; thus keep her i am not responding admiration, but there was done that band of course, my second evening.
Generating with 3-grams:
  Run 1: theatre under these hopeless proceedings

KeyError: (20433, 31950)

In [18]:
generate_sentence_smart(ngrams, 4, 250)

Generating with 4-grams:
  Run 1: why the jupiter of men.
  Run 2: poor mr.
  Run 3: peter . anon.
  Run 4: yet my detest and spurn thy to whom thou hast been where bell or diver never hast slept by many a many of of which the mind is not so free from shadow and unveiled its watch.
  Run 5: thou has a grim and thy face bears a command.
  Run 6: lucentio . come no time to learn to believe in the existence of happiness in being go to your anne gave a quick glance at the shop.
  Run 7: corruption in the text to venture on warning the telling him somehow that this companion in the there was no great harm done.
  Run 8: it was happy for me all my redeemed may dwell in joy and spent.
  Run 9: if i had been introduced to the and that friendship between a man who is his hands are cold as and when he runs he circles round to his rooms at once.
  Run 10: to bear the sight of and he ended with a of the there that single person was the exact embodiment of his utilitarian character.
  Run 11: maria